# Stage 4 — Readability & Faithfulness Analysis

**Reads:** `outputs/explanations.json`  
**Writes:** `outputs/results.csv`

Computes per-explanation metrics:

| Metric | Description |
|--------|-------------|
| `flesch_reading_ease` | Higher = simpler text |
| `flesch_kincaid_grade` | Higher = more complex |
| `smog_index` | Estimated years of education needed |
| `lime_coverage` | Fraction of LIME features referenced in explanation |
| `ontology_hit_rate` | Fraction of features mapped to ≥1 ancestor |

> **Tip:** This notebook is fast — safe to re-run freely as you tweak display cells.

## 1. Imports

In [1]:
import pandas as pd

from config import ANALYSIS_RESULTS_PATH, EXPLANATIONS_PATH
from pipeline_helpers import (
    checkpoint_exists,
    lime_coverage,
    load_checkpoint,
    ontology_hit_rate,
    readability_metrics,
    save_checkpoint,
)

## 2. Configuration

In [2]:
# Set True to recompute even if results.csv already exists
FORCE_RERUN = True

## 3. Load Stage 3 output

In [3]:
if not EXPLANATIONS_PATH.exists():
    raise FileNotFoundError(
        f"Explanations not found at '{EXPLANATIONS_PATH}'.\n"
        "Please run 03_llm.ipynb first."
    )
data = load_checkpoint(EXPLANATIONS_PATH)
print(f"Loaded {len(data)} explanations.")

[Checkpoint] Loaded 50 records ← 'outputs\explanations_expert.json'
Loaded 50 explanations.


## 4. Compute metrics

In [4]:
rows = []
for item in data:
    explanation  = item.get("explanation", "")
    feature_data = item.get("feature_data", [])

    row = {
        "text":              item["text"][:80] + "…",
        "predicted_class":   item["predicted_class"],
        "confidence":        item["confidence"],
        "user_category":     item["user_category"],
        "ablation_mode":     item["ablation_mode"],
        "explanation":       explanation,
        "lime_coverage":     lime_coverage(explanation, feature_data),
        "ontology_hit_rate": ontology_hit_rate(feature_data),
        **readability_metrics(explanation),
    }
    rows.append(row)

df = pd.DataFrame(rows)
print(f"✅ Metrics computed for {len(df)} explanations.")

✅ Metrics computed for 50 explanations.


## 5. Full results table

In [5]:
pd.set_option("display.max_colwidth", 60)
df[[
    "predicted_class", "user_category", "ablation_mode",
    "flesch_reading_ease", "flesch_kincaid_grade", "smog_index",
    "lime_coverage", "ontology_hit_rate"
]]

,predicted_class,user_category,ablation_mode,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
0,Digestive system diseases,EXPERT,normal,31.371604,13.901861,15.112258,1.0000,1.0
1,Cardiovascular diseases,EXPERT,normal,-11.430000,18.580000,17.122413,0.0000,0.0
2,Digestive system diseases,EXPERT,normal,13.299261,18.260957,19.487916,0.6667,1.0
3,Cardiovascular diseases,EXPERT,normal,20.231854,16.548715,17.353724,1.0000,1.0
4,Neoplasms,EXPERT,normal,38.386842,12.625263,15.579742,1.0000,1.0
5,Neoplasms,EXPERT,normal,22.802632,14.798947,16.218646,1.0000,1.0
6,General pathological conditions,EXPERT,normal,-10.330000,18.675000,18.243606,0.0000,0.0
7,Neoplasms,EXPERT,normal,40.841509,12.966063,14.191786,1.0000,1.0
8,General pathological conditions,EXPERT,normal,15.209565,16.851739,18.511140,1.0000,1.0
9,Digestive system diseases,EXPERT,normal,5.532500,16.462500,17.122413,0.0000,0.0


## 6. Summary — mean metrics by user category & ablation mode

In [6]:
metric_cols = [
    "flesch_reading_ease", "flesch_kincaid_grade",
    "smog_index", "lime_coverage", "ontology_hit_rate",
]
summary = (
    df.groupby(["user_category", "ablation_mode"])[metric_cols]
    .mean()
    .round(3)
)
summary

,,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
user_category,ablation_mode,,,,,
EXPERT,normal,14.244,16.219,16.802,0.633,0.64


## 7. [Optional] Per-class breakdown

In [7]:
df.groupby("predicted_class")[metric_cols].mean().round(3)

,flesch_reading_ease,flesch_kincaid_grade,smog_index,lime_coverage,ontology_hit_rate
predicted_class,,,,,
Cardiovascular diseases,7.017,17.226,17.087,0.667,0.667
Digestive system diseases,19.386,15.618,16.766,0.697,0.727
General pathological conditions,4.879,17.623,18.068,0.500,0.500
Neoplasms,23.082,14.860,15.965,0.692,0.692
Nervous system diseases,18.192,15.525,15.759,0.333,0.333


## 8. [Optional] Read a specific explanation in full

In [8]:
# Change the index to read any explanation in full
IDX = 0

row = df.iloc[IDX]
print(f"Text     : {row['text']}")
print(f"Class    : {row['predicted_class']} ({row['confidence']})")
print(f"User     : {row['user_category']}")
print(f"Ablation : {row['ablation_mode']}")
print(f"\n── Explanation ────────────────────────────────────")
print(row["explanation"])

Text     : Normalization of ventilation/perfusion relationships after liver transplantation…
Class    : Digestive system diseases (0.5919)
User     : EXPERT
Ablation : normal

── Explanation ────────────────────────────────────
assistant
The model predicted that the patient has a digestive system disease because the key feature identified as influential is the liver. The liver, which is part of the digestive system, plays a crucial role in various functions such as detoxification, metabolism, and bile production. When the liver is affected, it can lead to a variety of digestive system issues, such as liver cirrhosis, hepatitis, or gallstones. These conditions can cause symptoms like abdominal pain, jaundice, and changes in bowel habits. Therefore, the model attributes the patient's digestive symptoms to a liver-related issue, leading to the prediction of a digestive disease.


## 9. Save to CSV

In [9]:
df.to_csv(ANALYSIS_RESULTS_PATH, index=False)
print(f"✅ Saved {len(df)} rows → '{ANALYSIS_RESULTS_PATH}'")

✅ Saved 50 rows → 'outputs\results_expert_CD_2.csv'
